In [7]:
import pandas as pd
import os,nrrd,json,glob
import numpy as np
import shutil
from tqdm import tqdm

In [ ]:
# ["PL","ORB","ACA","MO","AI","ILA","FRP"]
area = "PFC"
outpath = r"J:\BLA_four_types\csv_%s"%area
os.makedirs(outpath,exist_ok=True)
files = glob.glob(r"J:\BLA_four_types\csv_terminal\*.csv")
for file in tqdm(files):
    filename = file.split("\\")[-1].split(".")[0]
    dfn = pd.read_csv(file,index_col=0)
    # dft = dfn.loc[(dfn["terminal"]==1)&(dfn["ML"]<=5700)&((dfn["region"]=="AM")|(dfn["region"]=="AD")|(dfn["region"]=="AV")|(dfn["region"]=="VTN"))]
    dft = dfn.loc[(dfn["terminal"]==1)&((dfn["name_use"].isin(["PL","ORB","ACA","ILA","FRP"])))]

In [ ]:
path_trajectories

In [ ]:
import os
import pandas as pd

def extract_pfc_subpathways_cleaned(
    file_path, 
    outpath, 
    target_regions=["PL", "ORB", "ACA", "ILA", "FRP"], 
    exclude_groups=["STR", "OLF",  "CTXsp", "amc"]  # 可在此处自由增删你想剔除的间隙/过渡脑区
):
    filename = os.path.basename(file_path).split(".")[0]
    dfn = pd.read_csv(file_path, index_col=0)
    
    # 1. 建立快速查找字典
    parent_map = dict(zip(dfn["ID"], dfn["parent"]))
    group_map = dict(zip(dfn["ID"], dfn["name_use"]))
    area_detail_map = dict(zip(dfn["ID"], dfn["area_name"]))
    
    # 2. 筛选目标末梢节点
    dft = dfn.loc[(dfn["terminal"] == 1) & (dfn["name_use"].isin(target_regions))]
    
    if dft.empty:
        print(f"未在 {filename} 中找到投射至 {target_regions} 的末梢节点。")
        return
    
    terminal_ids = dft["ID"].tolist()
    IDs = set()
    path_trajectories = []
    
    for tid in terminal_ids:
        curr = tid
        single_path_groups = []
        
        # 向上回溯直到胞体 (ID == 1)
        while curr != 1 and curr in parent_map:
            IDs.add(curr)
            single_path_groups.append(str(group_map.get(curr, "Unknown")))
            curr = parent_map[curr]
            
        IDs.add(1)
        single_path_groups.append(str(group_map.get(1, "Soma")))
        
        # 转为从 胞体 -> 终末 的正向顺序
        single_path_groups.reverse()
        
        # 【核心修改点】
        # 步骤 A：直接剔除用户指定的间隙/过渡脑区
        cleaned_groups = [g for g in single_path_groups if g not in exclude_groups]
        
        # 步骤 B：对过滤后的序列进行相邻连续去重
        compressed_path = []
        for g in cleaned_groups:
            if not compressed_path or compressed_path[-1] != g:
                compressed_path.append(g)
                
        path_trajectories.append({
            "Terminal_ID": tid,
            "Terminal_Area_Detail": area_detail_map.get(tid), # 细分亚区（如 ORBm1）
            "Terminal_Group": group_map.get(tid),             # 目标组（如 ORB）
            "Clean_Trajectory": " -> ".join(compressed_path)    # 精简后的核心路径
        })

    # 3. 提取子树 DataFrame
    dff = dfn.loc[dfn.ID.isin(IDs)].copy()
    dff["path_node_group"] = dff["name_use"]
    dff["is_pfc_target_terminal"] = dff["ID"].isin(terminal_ids)
    
    # 4. 保存结果
    os.makedirs(outpath, exist_ok=True)
    outfile = os.path.join(outpath, f"{filename}_PFC_subtree.csv")
    dff.to_csv(outfile)
    
    df_traj = pd.DataFrame(path_trajectories)
    traj_outfile = os.path.join(outpath, f"{filename}_PFC_clean_trajectories.csv")
    df_traj.to_csv(traj_outfile, index=False, encoding="utf-8-sig")
    
    print(f"处理完成: {filename}")
    print(f"- 过滤并精简后的路径表已保存至: {traj_outfile}\n")
    
    # 打印前 5 条精简后的路径示例
    print("=== 精简后的核心脑区投射路径示例 ===")
    for idx, row in df_traj.head(5).iterrows():
        print(f"末梢 [{row['Terminal_Area_Detail']}] (ID: {row['Terminal_ID']}):")
        print(f"  {row['Clean_Trajectory']}\n")

# 调用示例（你可以把不需要的脑区加进 exclude_groups 列表里）：
# extract_pfc_subpathways_cleaned("251034_003_Vglut1_PFC.csv", "./output_dir", exclude_groups=["STR", "OLF", "ec", "CTXsp", "amc", "FS"])

In [ ]:
extract_pfc_subpathways("251034_003_Vglut1_PFC.csv", "./output_dir")

In [ ]:
files = glob.glob(r"J:\BLA_four_types\csv_terminal\*.csv")
# os.makedirs(r"J:\BLA_four_types\csv_PFC_new")
for file in tqdm(files):
    extract_pfc_subpathways_cleaned(file, r"J:\BLA_four_types\csv_PFC_new")

In [ ]:
dfn.side == "left"

,ID,type,AP,DV,ML,R,parent,area_ID,area_name,name_use,bratch_ID,side,terminal
0,1,1,6454.62,5418.40,2393.60,11.332031,-1,579,ec,ec,1,left,0
1,2,2,6458.04,5421.70,2392.48,10.582031,1,579,ec,ec,1,left,0
2,3,2,6461.74,5425.34,2391.26,8.917969,2,579,ec,ec,1,left,0
3,4,2,6465.44,5428.94,2390.06,7.082031,3,131,LA,LA,1,left,0
4,5,2,6469.14,5432.52,2388.86,5.582031,4,131,LA,LA,1,left,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26429,26430,2,2072.00,2915.10,5562.64,1.000000,26429,484,ORBm1,ORB,212,left,0
26430,26431,2,2071.44,2913.46,5561.60,0.917969,26430,484,ORBm1,ORB,212,left,0
26431,26432,2,2071.64,2911.24,5561.04,1.167969,26431,484,ORBm1,ORB,212,left,0
26432,26433,2,2070.22,2907.70,5561.04,0.917969,26432,484,ORBm1,ORB,212,left,0


In [6]:
# all for PAG
# ["PL","ORB","ACA","MO","AI","ILA","FRP"]
areas = [["PB"],["AON"],["SI"]]
for area in areas:
    outpath = r"J:\BLA_three_types\terminal_new_name\csv_%s"%area[0]
    os.makedirs(outpath,exist_ok=True)
    files = glob.glob(r"J:\BLA_three_types\csv_terminal\*.csv")
    for file in tqdm(files):
        filename = file.split("\\")[-1].split(".")[0]
        dfn = pd.read_csv(file,index_col=0)
        # dft = dfn.loc[(dfn["terminal"]==1)&(dfn["ML"]<=5700)&((dfn["region"]=="AM")|(dfn["region"]=="AD")|(dfn["region"]=="AV")|(dfn["region"]=="VTN"))]
        dft = dfn.loc[(dfn["terminal"]==1)&((dfn["name_use"].isin(area)))&(dfn.side == "left")]
        try:
            last = dft.ID.tolist()[-1]
            IDlast = []
            IDlast.append(last)

            while last != 1:
                last = dfn.loc[dfn.ID==last].parent.values[0]
                IDlast.append(last)

            IDs = IDlast.copy()
            for IDt in dft.ID.tolist()[0:-1]:
                while IDt not in IDs:
                    IDs.append(IDt)
                    IDt = dfn.loc[dfn.ID==IDt].parent.values[0]

            dff = dfn.loc[dfn.ID.isin(IDs)]
            outfile = os.path.join(outpath,"%s_%s.csv"%(filename,area[0]))
            dff.to_csv(outfile)
            
        except:
            print(len(dft)) 
            print(file)     


  3%|▎         | 1/39 [00:01<00:54,  1.43s/it]

0
J:\BLA_three_types\csv_terminal\221058_040_Sst.csv
0
J:\BLA_three_types\csv_terminal\221058_042_Sst.csv
0
J:\BLA_three_types\csv_terminal\221058_071_Sst.csv
0
J:\BLA_three_types\csv_terminal\221058_088_Sst.csv
0
J:\BLA_three_types\csv_terminal\221058_104_Sst.csv
0
J:\BLA_three_types\csv_terminal\221058_107_Sst.csv
0
J:\BLA_three_types\csv_terminal\221058_113_Sst.csv


 28%|██▊       | 11/39 [00:06<00:22,  1.26it/s]

0
J:\BLA_three_types\csv_terminal\230058_073_Sst.csv


 38%|███▊      | 15/39 [00:13<00:34,  1.42s/it]

...

0
J:\BLA_three_types\csv_terminal\242655_057_Sst.csv
0
J:\BLA_three_types\csv_terminal\252459_074_Sst.csv
0
J:\BLA_three_types\csv_terminal\252459_079_Sst.csv
0
J:\BLA_three_types\csv_terminal\252460_132_Sst.csv
0
J:\BLA_three_types\csv_terminal\251034_001_Vglut1.csv


 92%|█████████▏| 36/39 [00:11<00:02,  1.09it/s]

0
J:\BLA_three_types\csv_terminal\251039_029_Vglut1.csv


100%|██████████| 39/39 [00:11<00:00,  3.30it/s]

0
J:\BLA_three_types\csv_terminal\251039_033_Vglut1.csv


In [8]:
# all for PAG
# ["PL","ORB","ACA","MO","AI","ILA","FRP"]
area = "PFC"
outpath = r"J:\BLA_three_types\flatmp\csv_%s"%area
os.makedirs(outpath,exist_ok=True)
files = glob.glob(r"J:\BLA_four_types\csv_terminal\*.csv")
for file in tqdm(files):
    filename = file.split("\\")[-1].split(".")[0]
    dfn = pd.read_csv(file,index_col=0)
    # dft = dfn.loc[(dfn["terminal"]==1)&(dfn["ML"]<=5700)&((dfn["region"]=="AM")|(dfn["region"]=="AD")|(dfn["region"]=="AV")|(dfn["region"]=="VTN"))]
    dft = dfn.loc[(dfn["terminal"]==1)&((dfn["name_use"].isin(["PL","ORB","ACA","ILA","FRP"])))]
    try:
        last = dft.ID.tolist()[-1]
        IDlast = []
        IDlast.append(last)

        while last != 1:
            last = dfn.loc[dfn.ID==last].parent.values[0]
            IDlast.append(last)

        IDs = IDlast.copy()
        for IDt in dft.ID.tolist()[0:-1]:
            while IDt not in IDs:
                IDs.append(IDt)
                IDt = dfn.loc[dfn.ID==IDt].parent.values[0]

        dff = dfn.loc[dfn.ID.isin(IDs)]
        outfile = os.path.join(outpath,"%s_%s.csv"%(filename,area))
        dff.to_csv(outfile)
        
    except:
        print(len(dft)) 
        print(file)     


  9%|▊         | 8/92 [00:00<00:02, 40.35it/s]

0
J:\BLA_four_types\csv_terminal\221058_038_Sst.csv
0
J:\BLA_four_types\csv_terminal\221058_040_Sst.csv
0
J:\BLA_four_types\csv_terminal\221058_042_Sst.csv
0
J:\BLA_four_types\csv_terminal\221058_071_Sst.csv
0
J:\BLA_four_types\csv_terminal\221058_088_Sst.csv
0
J:\BLA_four_types\csv_terminal\221058_104_Sst.csv
0
J:\BLA_four_types\csv_terminal\221058_107_Sst.csv
0
J:\BLA_four_types\csv_terminal\221058_113_Sst.csv
0
J:\BLA_four_types\csv_terminal\221297_036_Sst.csv
0
J:\BLA_four_types\csv_terminal\221297_037_Sst.csv
0
J:\BLA_four_types\csv_terminal\221297_038_Sst.csv
0
J:\BLA_four_types\csv_terminal\230058_073_Sst.csv


 21%|██        | 19/92 [00:00<00:01, 48.74it/s]

0
J:\BLA_four_types\csv_terminal\230058_074_Sst.csv
0
J:\BLA_four_types\csv_terminal\230058_076_Sst.csv
0
J:\BLA_four_types\csv_terminal\230058_078_Sst.csv
0
J:\BLA_four_types\csv_terminal\234064_011_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234064_014_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_001_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_007_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_008_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_009_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_010_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_015_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_016_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_018_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_019_Pkc.csv
0
J:\BLA_four_types\csv_terminal\234065_020_Pkc.csv


 39%|███▉      | 36/92 [00:00<00:00, 65.16it/s]

...

 96%|█████████▌| 88/92 [00:23<00:03,  1.19it/s]

0
J:\BLA_four_types\csv_terminal\251039_022_Vglut1.csv


 98%|█████████▊| 90/92 [00:36<00:03,  1.52s/it]

0
J:\BLA_four_types\csv_terminal\251039_031_Vglut1.csv


100%|██████████| 92/92 [00:38<00:00,  2.39it/s]


In [ ]:
dft = dfn.loc[(dfn["terminal"]==1)&((dfn["name_use"].isin(["BST"])))]

In [ ]:
dft

In [ ]:
# all for CA1
files = glob.glob(r"H:\20231030_figs\20231130_subtype4\dfbranch\*.csv")
for file in tqdm(files):
    filename = file.split("\\")[-1].split(".")[0]
    dfn = pd.read_csv(file,index_col=0)
    dft = dfn.loc[(dfn["terminal"]==1)&(dfn["ML"]<=5700)&(dfn["region"]=="CA1")]

    last = dft.ID.tolist()[-1]
    IDlast = []
    IDlast.append(last)

    while last != 1:
        last = dfn.loc[dfn.ID==last].parent.values[0]
        IDlast.append(last)

    IDs = IDlast.copy()
    for IDt in dft.ID.tolist()[0:-1]:
        while IDt not in IDs:
            IDs.append(IDt)
            IDt = dfn.loc[dfn.ID==IDt].parent.values[0]

    dff = dfn.loc[dfn.ID.isin(IDs)]
    dff.to_csv(".\\dfbrancth_CA1\\%s_CA1.csv"%filename)

In [ ]:
# all for CA1
files = glob.glob(r"H:\20231030_figs\20231130_subtype4\dfbranch\*.csv")
for file in tqdm(files):
    filename = file.split("\\")[-1].split(".")[0]
    dfn = pd.read_csv(file,index_col=0)
    dft = dfn.loc[(dfn["terminal"]==1)&(dfn["ML"]<=5700)&(dfn["region"]=="CA1")]
    try:
        last = dft.ID.tolist()[-1]
        IDlast = []
        IDlast.append(last)

        while last != 1:
            last = dfn.loc[dfn.ID==last].parent.values[0]
            IDlast.append(last)

        IDs = IDlast.copy()
        for IDt in dft.ID.tolist()[0:-1]:
            while IDt not in IDs:
                IDs.append(IDt)
                IDt = dfn.loc[dfn.ID==IDt].parent.values[0]

        dff = dfn.loc[dfn.ID.isin(IDs)]
        dff.to_csv(r"H:\20231030_figs\20231130_subtype4\dfbrancth_CA1\%s_CA1.csv"%filename)
    except:
        print(len(dft)) 
        print(file)     


In [ ]:
os.chdir(r"H:\20231030_figs\20231211_Subtype22")

In [ ]:
# all for SNr
files = glob.glob(r"H:\20231030_figs\20231211_Subtype22\dfbranch\*.csv")
for file in tqdm(files):
    filename = file.split("\\")[-1].split(".")[0]
    dfn = pd.read_csv(file,index_col=0)
    # dft = dfn.loc[(dfn["terminal"]==1)&(dfn["ML"]<=5700)&((dfn["region"]=="AM")|(dfn["region"]=="AD")|(dfn["region"]=="AV")|(dfn["region"]=="VTN"))]
    dft = dfn.loc[(dfn["terminal"]==1)&(dfn["ML"]<=5700)&((dfn["region"].isin(["AM","AD","AV","IAM","IAD","LD","MRN","CS","PCG","DTN","VTN"])))]
    try:
        last = dft.ID.tolist()[-1]
        IDlast = []
        IDlast.append(last)

        while last != 1:
            last = dfn.loc[dfn.ID==last].parent.values[0]
            IDlast.append(last)

        IDs = IDlast.copy()
        for IDt in dft.ID.tolist()[0:-1]:
            while IDt not in IDs:
                IDs.append(IDt)
                IDt = dfn.loc[dfn.ID==IDt].parent.values[0]

        dff = dfn.loc[dfn.ID.isin(IDs)]
        dff.to_csv(".\\df_ATN_VTN_others\\%s_ATN_VTN.csv"%filename)
    except:
        print(len(dft)) 
        print(file)     


In [ ]:
dfn = pd.read_csv(r"H:\20231030_figs\20231130_Subtype22\dfbranch\211269_044_Tac2_MMm.csv",index_col=0)
dft = dfn.loc[(dfn["terminal"]==1)&((dfn["region"]=="AM")|(dfn["region"]=="AD")|(dfn["region"]=="AV")|(dfn["region"]=="VTN"))]

In [ ]:
dft = dfn.loc[(dfn["region"]=="AM")]

In [ ]:
dft = dfn.loc[(dfn["region"]=="AM")&(dfn["terminal"]==1)]

In [ ]:
dft

In [ ]:
dfn = pd.read_csv("211460_010_Nts_STN.csv",index_col=0)

In [ ]:
dft = dfn.loc[(dfn["terminal"]==1)&(dfn["region"]=="SNr")]

In [ ]:
dfn

In [ ]:
dft.loc[dft.ID==28928].parent.values[0]

In [ ]:
dft.ID.tolist()[-1]

In [ ]:
dft

In [ ]:
last = dft.ID.tolist()[-1]
IDlast = []
IDlast.append(last)

while last != 1:
    last = dfn.loc[dfn.ID==last].parent.values[0]
    IDlast.append(last)


In [ ]:
IDs = IDlast.copy()
for IDt in dft.ID.tolist()[0:-1]:
    while IDt not in IDs:
        IDs.append(IDt)
        IDt = dfn.loc[dfn.ID==IDt].parent.values[0]


In [ ]:
dfn.loc[dfn.ID.isin(IDs)]

In [ ]:
len(IDlast)

In [ ]:
aa

In [ ]:
dft.loc[dft.ID==28927].parent.values[0]

In [ ]:
last = dft.ID.tolist()[-1]
IDlast = []
IDlast.append(last)

while last != 1:
    last = dft.loc[dft.ID==last].parent.values[0]
    IDlast.append(last)



for IDt in dft.ID.tolist():
    IDs = []
    IDs.append(IDt)
    IDtmp = IDt
    while IDtmp != 1:
        IDtmp = dft.loc[dft.ID==IDtmp].parent.values[0]
        IDs.append(IDtmp)

    
    print(IDt)